# Automated ML Data Preprocessor
**GetIntern.in | Week 5 Milestone Project | Phase 2: Predictive Modeling & Feature Prep**

This notebook builds an automated preprocessing pipeline that turns the Telco Customer Churn dataset (mixed text and numeric columns) into clean, scaled, model-ready train and test sets using **Pandas and scikit-learn**.

**Objectives**
1. **Categorical feature encoding** with `LabelEncoder` and `pd.get_dummies()`
2. **Z-score feature scaling** with `StandardScaler`
3. **Train-test partitioning** (80/20) with `train_test_split(test_size=0.2, random_state=42)`

**Note on order:** the split (Objective 3) is done **before** scaling (Objective 2). The scaler learns its mean and standard deviation from the training data only, which prevents data leakage from the test set.

**Exports:** `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` for model training in Week 6.

## Setup
### Step 1: Import libraries
Pandas and NumPy handle the data. scikit-learn provides `LabelEncoder`, `StandardScaler` and `train_test_split`.

In [53]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

### Step 2: Load the dataset
Load `WA_Fn-UseC_-Telco-Customer-Churn.csv`: 7,043 customers and 21 columns. The raw shape is saved so it can be compared with the final shapes at the end.

In [54]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
raw_shape = df.shape
print("Raw shape:", raw_shape)
df.head()

Raw shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Step 3: Inspect the data
Check data types, duplicates, the target (`Churn`: 5,174 stayed and 1,869 churned) and which columns are text and which are numeric. `TotalCharges` is stored as text and needs cleaning.

In [55]:
df.info()
print("\nDuplicates:", df.duplicated().sum())
print("\nChurn:\n", df["Churn"].value_counts())
print("\nText columns:", df.select_dtypes(include="object").columns.tolist())
print("\nNumeric columns:", df.select_dtypes(include="number").columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


### Step 4: Clean the data
- Remove duplicate rows (none were found).
- Convert `TotalCharges` to numbers. The 11 blank values belong to new customers with tenure 0 who have not been billed yet, so they are filled with 0.
- Strip extra spaces from text columns.
- Drop `customerID`, because it is a unique label and carries no predictive information.

In [56]:
df = df.drop_duplicates().reset_index(drop=True)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)   # 11 blanks = new customers
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()
df = df.drop(columns="customerID")      # an ID, not a predictive feature

print("Shape after cleaning:", df.shape, "| Nulls:", df.isnull().sum().sum())

Shape after cleaning: (7043, 20) | Nulls: 0


## Objective 1: Categorical Feature Encoding
### Step 5: Encode the target variable
`LabelEncoder` converts the target `Churn` from text to numbers: **No = 0** and **Yes = 1**. The remaining columns become the feature matrix `X`, and the target is stored separately as `y`.

In [57]:
le_target = LabelEncoder()
y = pd.Series(le_target.fit_transform(df["Churn"]), name="Churn")
print(dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))
print(y.value_counts())

X = df.drop(columns="Churn")

{'No': np.int64(0), 'Yes': np.int64(1)}
Churn
0    5174
1    1869
Name: count, dtype: int64


### Step 6: Group the columns by type
Each type of column needs a different treatment:
- **Binary text columns** (two values, for example Yes/No): encoded with `LabelEncoder`.
- **Multi-category text columns** (three or more values, for example Contract): encoded with `pd.get_dummies()`.
- **Numeric columns to scale:** `tenure`, `MonthlyCharges` and `TotalCharges`.

In [58]:
cat_cols = X.select_dtypes(include="object").columns.tolist()
binary_cols = [c for c in cat_cols if X[c].nunique() == 2]
multi_cols = [c for c in cat_cols if X[c].nunique() > 2]
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

print("Binary:", binary_cols)
print("Multi-category:", multi_cols)
print("Numeric to scale:", num_cols)

Binary: ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
Multi-category: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
Numeric to scale: ['tenure', 'MonthlyCharges', 'TotalCharges']


### Step 7: Encode binary columns with `LabelEncoder`
Columns with only two values, such as `gender`, `Partner`, `Dependents`, `PhoneService` and `PaperlessBilling`, are converted to 0 and 1. The mapping for each column is saved so the encoding can be reversed or reused.

In [59]:
encoders = {}
for col in binary_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = dict(zip(le.classes_, le.transform(le.classes_)))
encoders

{'gender': {'Female': np.int64(0), 'Male': np.int64(1)},
 'Partner': {'No': np.int64(0), 'Yes': np.int64(1)},
 'Dependents': {'No': np.int64(0), 'Yes': np.int64(1)},
 'PhoneService': {'No': np.int64(0), 'Yes': np.int64(1)},
 'PaperlessBilling': {'No': np.int64(0), 'Yes': np.int64(1)}}

### Step 8: Encode multi-category columns with `pd.get_dummies()`
Columns with three or more values (for example `Contract` and `PaymentMethod`) are turned into separate 0/1 flag columns.
- **Why not `LabelEncoder` here?** It would create a false order (for example Month-to-month = 0, One year = 1, Two year = 2), which a model could misread as a ranking.
- `drop_first=True` removes one redundant column per feature, because it can be inferred from the others.

In [60]:
X = pd.get_dummies(X, columns=multi_cols, drop_first=True, dtype=int)
print("Shape after encoding:", X.shape)
X.head()

Shape after encoding: (7043, 30)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,MultipleLines_No phone service,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,1,...,0,0,0,0,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,0,0,0,1,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,0,...,0,0,0,0,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,1,...,1,0,0,0,0,1,0,0,0,0
4,0,0,0,0,2,1,1,70.70,151.65,0,...,0,0,0,0,0,0,0,0,1,0


### Step 9: Verify that everything is numeric
Confirm that no text columns remain and no values are missing. The feature matrix is now fully machine-readable.

In [61]:
print("Non-numeric columns:", X.select_dtypes(exclude="number").columns.tolist())
print("Nulls:", X.isnull().sum().sum())

Non-numeric columns: []
Nulls: 0


## Objective 3: Train-Test Partitioning (done before scaling to prevent data leakage)
### Step 10: 80/20 split
`train_test_split(test_size=0.2, random_state=42)` splits the data into 80% for training and 20% for testing. The fixed `random_state=42` makes the split reproducible.
- **Training set:** 5,634 customers
- **Test set:** 1,409 customers
The churn rate is almost the same in both parts, so the split is representative.

In [62]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_test = X_train.copy(), X_test.copy()

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)
print("Churn rate train:", round(y_train.mean() * 100, 2), "| test:", round(y_test.mean() * 100, 2))

X_train: (5634, 30) | X_test: (1409, 30)
y_train: (5634,) | y_test: (1409,)
Churn rate train: 26.55 | test: 26.47


## Objective 2: Z-Score Feature Scaling (fitted on training data only)
### Step 11: Apply `StandardScaler`
`StandardScaler` converts each numeric column to a z-score: (value − mean) ÷ standard deviation. This gives every column a mean of 0 and a standard deviation of 1, so large-valued columns like `TotalCharges` do not dominate small-valued ones like `tenure`.
- `fit_transform` is applied to the **training set**, so the scaler learns the mean and standard deviation from training data only.
- `transform` is applied to the **test set** using the same values, so no information from the test set leaks into training.

In [63]:
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])   # learn from train only
X_test[num_cols] = scaler.transform(X_test[num_cols])         # apply to test

### Step 12: Check the scaling
The scaled training columns should have a mean of about 0 and a standard deviation of about 1. The test columns will be close to these values but not exact, which is expected, because they were scaled with the training set's statistics.

In [64]:
pd.DataFrame({
    "train_mean": X_train[num_cols].mean().round(4),
    "train_std": X_train[num_cols].std(ddof=0).round(4),
    "test_mean": X_test[num_cols].mean().round(4),
    "test_std": X_test[num_cols].std(ddof=0).round(4),
})

,train_mean,train_std,test_mean,test_std
tenure,0.0,1.0,-0.0004,1.0274
MonthlyCharges,-0.0,1.0,-0.0170,1.0001
TotalCharges,0.0,1.0,-0.0162,1.0079


### Step 13: Scaling summary
The original mean and standard deviation that the scaler learned from the training data for each numeric column. These values are needed to scale any new customer data in the same way.

In [65]:
pd.DataFrame({"original_mean": scaler.mean_, "original_std": np.sqrt(scaler.var_)}, index=num_cols).round(2)

,original_mean,original_std
tenure,32.37,24.42
MonthlyCharges,64.86,30.09
TotalCharges,2287.09,2263.00


## Before and After Summary
### Step 14: Shape comparison table
Shows how the data changed at each stage, from the raw dataset to the final train and test files.

In [66]:
shape_table = pd.DataFrame({
    "Stage": ["Raw dataset", "After cleaning (ID dropped)", "Encoded features (X)", "X_train", "X_test", "y_train", "y_test"],
    "Rows": [raw_shape[0], len(df), X.shape[0], X_train.shape[0], X_test.shape[0], len(y_train), len(y_test)],
    "Columns": [raw_shape[1], df.shape[1], X.shape[1], X_train.shape[1], X_test.shape[1], 1, 1],
})
shape_table

,Stage,Rows,Columns
0,Raw dataset,7043,21
1,After cleaning (ID dropped),7043,20
2,Encoded features (X),7043,30
3,X_train,5634,30
4,X_test,1409,30
5,y_train,5634,1
6,y_test,1409,1


### Step 15: Feature table
Shows what happened to each original column: which method was used and how many columns it produced. The total of 30 feature columns matches the number of columns in `X`, which confirms that nothing was lost.

In [67]:
rows = []
for col in binary_cols:
    rows.append({"original_column": col, "type": "binary", "method": "LabelEncoder", "new_columns": 1})
for col in multi_cols:
    n = len([c for c in X.columns if c.startswith(col + "_")])
    rows.append({"original_column": col, "type": "multi-category", "method": "get_dummies (drop_first)", "new_columns": n})
for col in num_cols:
    rows.append({"original_column": col, "type": "numeric", "method": "StandardScaler (z-score)", "new_columns": 1})
rows.append({"original_column": "SeniorCitizen", "type": "already 0/1", "method": "kept as is", "new_columns": 1})

feature_table = pd.DataFrame(rows)
print("Total features:", feature_table["new_columns"].sum(), "| X columns:", X.shape[1])
feature_table

Total features: 30 | X columns: 30


,original_column,type,method,new_columns
0,gender,binary,LabelEncoder,1
1,Partner,binary,LabelEncoder,1
2,Dependents,binary,LabelEncoder,1
3,PhoneService,binary,LabelEncoder,1
4,PaperlessBilling,binary,LabelEncoder,1
5,MultipleLines,multi-category,get_dummies (drop_first),2
6,InternetService,multi-category,get_dummies (drop_first),2
7,OnlineSecurity,multi-category,get_dummies (drop_first),2
8,OnlineBackup,multi-category,get_dummies (drop_first),2
9,DeviceProtection,multi-category,get_dummies (drop_first),2


### Step 16: Tables in markdown format
Prints both tables as markdown so they can be copied straight into the README.

In [68]:
print(shape_table.to_markdown(index=False))
print()
print(feature_table.to_markdown(index=False))

| Stage                       |   Rows |   Columns |
|:----------------------------|-------:|----------:|
| Raw dataset                 |   7043 |        21 |
| After cleaning (ID dropped) |   7043 |        20 |
| Encoded features (X)        |   7043 |        30 |
| X_train                     |   5634 |        30 |
| X_test                      |   1409 |        30 |
| y_train                     |   5634 |         1 |
| y_test                      |   1409 |         1 |

| original_column   | type           | method                   |   new_columns |
|:------------------|:---------------|:-------------------------|--------------:|
| gender            | binary         | LabelEncoder             |             1 |
| Partner           | binary         | LabelEncoder             |             1 |
| Dependents        | binary         | LabelEncoder             |             1 |
| PhoneService      | binary         | LabelEncoder             |             1 |
| PaperlessBilling  | binary  

## Export
### Step 17: Save the processed partitions
Export the four files for model training in Week 6:
- `X_train.csv` and `X_test.csv` (features)
- `y_train.csv` and `y_test.csv` (target labels)

In [69]:
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)
print("Saved 4 files")

Saved 4 files


### Step 18: Verify the exported files
Reload each file and confirm its shape matches the tables above.

In [70]:
for name in ["X_train", "X_test", "y_train", "y_test"]:
    print(name, pd.read_csv(f"{name}.csv").shape)

X_train (5634, 30)
X_test (1409, 30)
y_train (5634, 1)
y_test (1409, 1)


## Conclusion
- **Cleaned** the Telco data (7,043 rows) and removed the ID column.
- **Encoded** 5 binary columns with `LabelEncoder` and 10 multi-category columns with `pd.get_dummies()`, producing a fully numeric matrix of **30 features**.
- **Split** the data 80/20 (5,634 train and 1,409 test rows) with `random_state=42`.
- **Scaled** `tenure`, `MonthlyCharges` and `TotalCharges` with `StandardScaler`, fitted on the training set only to avoid data leakage.
- **Exported** `X_train.csv`, `X_test.csv`, `y_train.csv` and `y_test.csv`, ready for model training in Week 6.